🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
🧩 Scenario
You are an AI intern at an ed-tech company. Students frequently ask questions about:
Course policies (refunds, deadlines)
Lecture content (PDF notes)
Assignment deadlines
Their enrollment status
The company wants a single intelligent assistant that:
Answers questions using internal documents (PDFs, FAQs)
Fetches student-specific data (like enrollment or progress) using tools/APIs
Avoids hallucination and gives reliable answers
💻 Task
Design and implement a working prototype (pseudo-code or real code) for this system.
Your solution must include:
✅ 1. RAG Pipeline
How you will ingest and preprocess documents
Chunking strategy (with justification)
Embedding + vector store choice
Retrieval logic
How context is injected into the LLM
✅ 2. Agent Integration
Design an agent that decides:
When to use RAG
When to call a tool (e.g., get_student_status(student_id))
Show how tools are defined and used
✅ 3. End-to-End Flow
Write code or structured pseudo-code showing:
Input query
Retrieval step
Tool calling (if needed)
Final answer generation
✅ 4. Reliability Improvements
Show at least 2 techniques in code/design to:
Reduce hallucination
Improve answer quality
🎯 Expected Outcome
A working architecture/code that demonstrates:
RAG + Agent working together
Decision-making capability
Real-world practicality

In [8]:
!!pip install langchain langgraph faiss-cpu sentence-transformers langchain-community pypdf langchain-text-splitters langchain-core -q

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
# ── Dummy LLM ─────────────────────────────────────────────────────────────────
class DummyLLM:
    def invoke(self, prompt):
        return "Generated answer based on provided context."

llm = DummyLLM()

# ── 1. DOCUMENT INGESTION ─────────────────────────────────────────────────────
def load_documents():
    try:
        docs = PyPDFLoader("lecture_notes.pdf").load()
        print("Loaded PDF.")
        return docs
    except Exception:
        pass
    try:
        docs = TextLoader("faq.txt").load()
        print("Loaded TXT.")
        return docs
    except Exception:
        pass
    # Fallback synthetic docs
    print("No files found — using fallback documents.")
    return [
        Document(page_content="Deep learning covers neural networks and backpropagation.", metadata={}),
        Document(page_content="The next assignment deadline is 25 March.", metadata={}),
        Document(page_content="Student 101 is enrolled and has 70% progress.", metadata={}),
    ]

# ── 2 & 3. CHUNKING + EMBEDDINGS + VECTOR STORE ───────────────────────────────
def build_vectorstore():
    docs = load_documents()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embedding)
    return vectorstore

vectorstore = build_vectorstore()
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ── 4. TOOLS ──────────────────────────────────────────────────────────────────
def get_student_status(student_id):
    return f"Student {student_id}: Enrolled, Progress 70%"

def get_assignment_deadlines():
    return "Next deadline: 25 March - Deep Learning Assignment"

# ── 5. AGENT ROUTER ───────────────────────────────────────────────────────────
def agent_router(query):
    q = query.lower()
    if "status" in q or "progress" in q:
        return "student_tool"
    elif "deadline" in q:
        return "deadline_tool"
    return "rag"

# ── 6. RAG PIPELINE ───────────────────────────────────────────────────────────
def rag_pipeline(query):
    docs = retriever.invoke(query)
    if not docs:
        return "I don't know (no relevant information found)."
    context = "\n".join([doc.page_content for doc in docs])
    prompt = f"""You are an AI assistant.
Answer ONLY using the given context.

Context:
{context}

Question:
{query}

If the answer is not in the context, say "I don't know".
"""
    return llm.invoke(prompt)

# ── 7. END-TO-END ─────────────────────────────────────────────────────────────
def run_rag_agent(query, student_id="101"):
    decision = agent_router(query)
    if decision == "student_tool":
        return get_student_status(student_id)
    elif decision == "deadline_tool":
        return get_assignment_deadlines()
    return rag_pipeline(query)

# ── TEST ──────────────────────────────────────────────────────────────────────
print(run_rag_agent("What is my progress?"))
print(run_rag_agent("When is the next deadline?"))
print(run_rag_agent("Explain neural networks"))

/tmp/ipykernel_1268/4121121683.py:42: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loaded PDF.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Student 101: Enrolled, Progress 70%
Next deadline: 25 March - Deep Learning Assignment
Generated answer based on provided context.
